
### Question - 011: - Write a pyspark query using below input data tables to get provided output data table

Input Table : 1
|    player| runs|50s/100s|
|----------|-----|--------|
|Sachin-IND|18694|   93/49|
| Ricky-AUS|11274|   66/31|
|   Lara-WI|10222|   45/21|
| Rahul-IND|10355|   95/11|
| Jhonty-SA| 7051|    43/5|
|Hayden-AUS| 8722|   67/19|

Input Table : 2
|country_code|country_name|
|------------|------------|
|         IND|       India|
|         AUS|   Australia|
|          WI|  WestIndies|
|          SA| SouthAfrica|

Output Table : 3

|player_name|country_name| runs|total_boundries|
|-----------|------------|-----|---------------|
|      Ricky|   Australia|11274|             97|
|     Hayden|   Australia| 8722|             86|
|     Sachin|       India|18694|            142|
|      Rahul|       India|10355|            106|
|     Jhonty| SouthAfrica| 7051|             48|
|       Lara|  WestIndies|10222|             66|


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("spark-q-0011").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 16:18:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
from pyspark.sql.types import *

schema = StructType([
    StructField("player", StringType(), True),
    StructField("runs", IntegerType(), True),
    StructField("50s/100s", StringType(), True)
])

# Create a DataFrame with the defined schema
players = [("Sachin-IND", 18694, "93/49"), 
           ("Ricky-AUS", 11274, "66/31"),
           ("Lara-WI", 10222, "45/21"),
           ("Rahul-IND", 10355, "95/11"),
           ("Jhonty-SA", 7051, "43/5"),
           ("Hayden-AUS", 8722, "67/19")
           ]
players_df = spark.createDataFrame(players, schema)
players_df.show()


counties = [("IND", "India"), 
            ("AUS", "Australia"), 
            ("WI", "WestIndies"), 
            ("SA", "SouthAfrica")
        ]
countries_df = spark.createDataFrame(counties,["country_code","country_name"])
countries_df.show()



+----------+-----+--------+
|    player| runs|50s/100s|
+----------+-----+--------+
|Sachin-IND|18694|   93/49|
| Ricky-AUS|11274|   66/31|
|   Lara-WI|10222|   45/21|
| Rahul-IND|10355|   95/11|
| Jhonty-SA| 7051|    43/5|
|Hayden-AUS| 8722|   67/19|
+----------+-----+--------+

+------------+------------+
|country_code|country_name|
+------------+------------+
|         IND|       India|
|         AUS|   Australia|
|          WI|  WestIndies|
|          SA| SouthAfrica|
+------------+------------+



In [ ]:
from pyspark.sql.functions import *

# split columns
split_nm_col = split(col("player"), "-")
split_bou_col = split(col("50s/100s"), "/")


result_df = (
    players_df
    .withColumn("player_name", split_nm_col[0])
    .withColumn("country_code", split_nm_col[1])
    .withColumn("fifties", split_bou_col[0])
    .withColumn("hundereds", split_bou_col[1])
    .drop("player","50s/100s"))

result_df = result_df.join(countries_df, result_df.country_code == countries_df.country_code).drop(countries_df.country_code)

result_df.show()

+-----+-----------+------------+-------+---------+------------+
| runs|player_name|country_code|fifties|hundereds|country_name|
+-----+-----------+------------+-------+---------+------------+
|11274|      Ricky|         AUS|     66|       31|   Australia|
| 8722|     Hayden|         AUS|     67|       19|   Australia|
|18694|     Sachin|         IND|     93|       49|       India|
|10355|      Rahul|         IND|     95|       11|       India|
| 7051|     Jhonty|          SA|     43|        5| SouthAfrica|
|10222|       Lara|          WI|     45|       21|  WestIndies|
+-----+-----------+------------+-------+---------+------------+



In [13]:
final_df = (
            result_df
            .withColumn("total_boundries", (col("fifties").cast("int") + col("hundereds").cast("int")))
        ).select("player_name","country_name", "runs", "total_boundries")
final_df.show()

+-----------+------------+-----+---------------+
|player_name|country_name| runs|total_boundries|
+-----------+------------+-----+---------------+
|      Ricky|   Australia|11274|             97|
|     Hayden|   Australia| 8722|             86|
|     Sachin|       India|18694|            142|
|      Rahul|       India|10355|            106|
|     Jhonty| SouthAfrica| 7051|             48|
|       Lara|  WestIndies|10222|             66|
+-----------+------------+-----+---------------+

